# Spike Removal Demo — BNO085 IMU

**Goal:** remove spike-type outliers from raw IMU streams before feeding them to
downstream ML. Keep the pipeline small, transparent, and loss-less for normal
samples.

### Pipeline (per channel)

1. **Mask full-scale outliers** &nbsp;&nbsp;`|v| > FS` &nbsp;→&nbsp; impossible physically (acquisition bug)
2. **Estimate bias** &nbsp;&nbsp;`bias = median(v)`&nbsp;&nbsp; (accel-z subtracts `g = 9.81` first)
3. **Compute z-score** &nbsp;&nbsp;`z = (v − bias) / σ_ref`
4. **Mask spikes** &nbsp;&nbsp;`|z| > 5` &nbsp;→&nbsp; probability under Gaussian ≈ 5.7 × 10⁻⁷
5. **Linear interpolation** to fill the masked positions

### Why σ_ref is fixed across runs

`σ_ref` is derived **once** from the clean healthy runs (Run008 + Run009) using
**MAD × 1.4826** (robust against the few existing spikes). Once fixed, it is
reused as the unit of scale for all runs — this lets the z-score threshold
`|z| > 5` have the **same physical meaning** in every run, including polluted
ones (Run0011 / Run0012) where the naive in-run std is unusable.

### Outputs

- σ_ref table + σ_ref vs LSB bar chart
- **Plot group A** – raw + spike-dots + clean (overlay) per run
- **Plot group B** – clean-only 3×3 grid per run (no spike markers)
- Zoom-in detail on Run0011 first 2 s

In [ ]:
import os, sys
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

plt.rcParams.update({
    'figure.dpi': 110, 'font.size': 10, 'axes.grid': True, 'grid.alpha': 0.3,
    'axes.titlesize': 10, 'axes.labelsize': 9, 'lines.linewidth': 0.7,
})

# ---- constants ------------------------------------------------------------
# BNO085 physical full-scale limits (SI units)
BNO085_FS = {
    'gyro':  34.9066,   # +/- 2000 dps
    'accel': 156.9060,  # +/- 16 g
    'mag':   1300.0,    # +/- 1300 uT
}
# BNO085 Q-point quantization step (1 LSB in SI units)
BNO085_LSB = {
    'gyro':  1.0 / 512.0,   # Q9
    'accel': 1.0 / 256.0,   # Q8
    'mag':   1.0 / 16.0,    # Q4
}
GRAVITY = 9.81  # m/s^2, used to offset accel-z bias before median

# Runs to process. Run008/009 are the healthy references; Run0011/0012 are
# polluted; Run001 is an extra healthy run for sanity-check.
RUNS = ['001', '008', '009', '0011', '0012']
REF_RUNS = ['008', '009']  # used to estimate sigma_ref

# Data root template. Adjust here when moving off Google Drive.
DATA_ROOT_FMT = "/content/drive/MyDrive/capstone_data/UNIT_0001_RUN_{run}"

SENSOR_FILES = {
    'gyro':  'gyroscope.csv',
    'accel': 'accelerometer.csv',
    'mag':   'magnetometer.csv',
}
AXES = ['x', 'y', 'z']
COLORS = {'x': '#E74C3C', 'y': '#2ECC71', 'z': '#3498DB'}


# ---- google-drive mount (no-op outside Colab) -----------------------------
IN_COLAB = 'google.colab' in sys.modules
if IN_COLAB:
    try:
        from google.colab import drive
        if not os.path.exists('/content/drive/MyDrive'):
            drive.mount('/content/drive')
    except Exception as e:
        print(f"[warn] drive mount skipped: {e}")


# ---- CSV loader -----------------------------------------------------------
def load_sensor(path: str) -> pd.DataFrame:
    """Load a single sensor CSV and normalize columns to [t_ms, x, y, z, time_s]."""
    df = pd.read_csv(path)
    cols = {c.lower(): c for c in df.columns}
    tcol = next(
        (cols[k] for k in cols if 'time' in k or 'ts' in k or k == 't'),
        df.columns[0],
    )
    xyz = []
    for k in AXES:
        cand = [c for c in df.columns if c.lower() == k or c.lower().endswith('_' + k)]
        if cand:
            xyz.append(cand[0])
    if len(xyz) < 3:
        xyz = [c for c in df.columns if c != tcol][:3]
    df = df[[tcol] + xyz].copy()
    df.columns = ['t_ms', 'x', 'y', 'z']
    for c in df.columns:
        df[c] = pd.to_numeric(df[c], errors='coerce')
    df['time_s'] = (df['t_ms'] - df['t_ms'].iloc[0]) / 1000.0
    return df


print("Setup complete.")
print(f"  RUNS     = {RUNS}")
print(f"  REF_RUNS = {REF_RUNS}  (used to estimate sigma_ref)")

In [ ]:
# Load all 9 channels for every run into DATA[run][sensor] = DataFrame.
# Missing files are skipped with a warning rather than a hard failure.

DATA = {}

print("=" * 70)
print("  Loading CSVs")
print("=" * 70)

for run in RUNS:
    data_root = DATA_ROOT_FMT.format(run=run)
    DATA[run] = {}
    print(f"\n  Run {run}   ({data_root})")
    for sensor, fn in SENSOR_FILES.items():
        path = os.path.join(data_root, fn)
        if not os.path.exists(path):
            print(f"    [skip] {sensor:<5} not found")
            continue
        df = load_sensor(path)
        DATA[run][sensor] = df
        print(f"    {sensor:<5}  rows={len(df):>9,}  duration={df['time_s'].iloc[-1]:.1f}s")

In [ ]:
# Estimate sigma_ref per (sensor, axis) from the healthy reference runs.
#
# Robust estimator: sigma_ref = 1.4826 * MAD(v - median(v))
# MAD = Median Absolute Deviation. The 1.4826 factor converts MAD to the
# equivalent Gaussian standard deviation. MAD is immune to the handful of
# spikes that may still live inside even the "clean" runs, so this is the
# number we can trust as a cross-run scale.
#
# For accel-z we subtract gravity before computing MAD, otherwise we'd be
# characterising the bias magnitude instead of the noise spread.

def mad_sigma(v: np.ndarray) -> float:
    """Robust std estimate via Median Absolute Deviation."""
    v = np.asarray(v, dtype=float)
    v = v[np.isfinite(v)]
    med = np.median(v)
    return 1.4826 * np.median(np.abs(v - med))


print("=" * 70)
print("  Estimating sigma_ref from reference runs (Run" +
      " + Run".join(REF_RUNS) + ")")
print("=" * 70)

SIGMA_REF = {}  # SIGMA_REF[sensor][axis] = float

for sensor in SENSOR_FILES:
    SIGMA_REF[sensor] = {}
    for axis in AXES:
        pooled = []
        for run in REF_RUNS:
            if sensor not in DATA.get(run, {}):
                continue
            v = DATA[run][sensor][axis].to_numpy()
            # drop full-scale outliers first
            v = v[np.abs(v) <= BNO085_FS[sensor]]
            # remove gravity on accel-z so we measure spread, not offset
            if sensor == 'accel' and axis == 'z':
                v = v - GRAVITY
            pooled.append(v)
        pooled = np.concatenate(pooled) if pooled else np.array([])
        SIGMA_REF[sensor][axis] = mad_sigma(pooled) if pooled.size else np.nan

# Pretty-print as a DataFrame.
sigma_tbl = pd.DataFrame(SIGMA_REF).T
sigma_tbl.columns = [f"sigma_{a}" for a in AXES]
sigma_tbl['LSB'] = [BNO085_LSB[s] for s in sigma_tbl.index]
sigma_tbl['sigma/LSB (x)'] = sigma_tbl['sigma_x'] / sigma_tbl['LSB']
print(sigma_tbl.round(6))


# Bar chart: sigma_ref per channel vs the sensor's LSB (both in SI units).
fig, ax = plt.subplots(figsize=(8, 3.2))
labels = [f"{s}-{a}" for s in SENSOR_FILES for a in AXES]
sigmas = [SIGMA_REF[s][a] for s in SENSOR_FILES for a in AXES]
lsbs   = [BNO085_LSB[s]    for s in SENSOR_FILES for a in AXES]
xp = np.arange(len(labels))
ax.bar(xp - 0.2, sigmas, width=0.4, label='sigma_ref', color='#3498DB')
ax.bar(xp + 0.2, lsbs,   width=0.4, label='1 LSB',     color='#95A5A6')
ax.set_yscale('log')
ax.set_xticks(xp)
ax.set_xticklabels(labels, rotation=30)
ax.set_ylabel('SI units (log scale)')
ax.set_title('sigma_ref vs 1 LSB per channel')
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
# Core cleaning routine for a single channel.
#
# Returns:
#   v_clean    : np.ndarray, same length as v, spikes replaced by linear interp
#   mask_spike : np.ndarray[bool], True where a spike was detected
#   z          : np.ndarray, z-score of the raw signal against (bias, sigma_ref)
#   bias       : float, estimated constant offset
#
# Mask policy:
#   - A sample is "bad" if (a) |v| > FS (physically impossible) OR
#                          (b) |z| > 5  (spike beyond 5-sigma)
#   - Bad samples are replaced via linear interpolation based on sample index.
#   - Leading/trailing NaNs are back-/forward-filled so the output has no gaps.

SPIKE_Z_THRESHOLD = 5.0


def clean_channel(v: np.ndarray, sensor: str, axis: str, sigma_ref: float):
    v = np.asarray(v, dtype=float)

    # (1) full-scale mask
    mask_fs = np.abs(v) > BNO085_FS[sensor]

    # (2) robust bias on the survivors
    good_for_bias = v[~mask_fs & np.isfinite(v)]
    bias = np.median(good_for_bias) if good_for_bias.size else 0.0
    if sensor == 'accel' and axis == 'z':
        # subtract gravity so the bias represents the non-physical offset only
        bias = bias - GRAVITY

    # (3) z-score against the FIXED cross-run sigma_ref
    if sensor == 'accel' and axis == 'z':
        centered = v - (bias + GRAVITY)
    else:
        centered = v - bias
    z = centered / sigma_ref

    # (4) spike mask (combine with FS mask)
    mask_spike = np.abs(z) > SPIKE_Z_THRESHOLD
    mask_bad = mask_fs | mask_spike | ~np.isfinite(v)

    # (5) linear interpolation
    s = pd.Series(v.copy())
    s[mask_bad] = np.nan
    s = s.interpolate(method='linear', limit_direction='both')
    v_clean = s.to_numpy()

    return v_clean, mask_spike, z, bias


print("clean_channel() ready.")

In [ ]:
# Run the cleaning pipeline on every run/sensor/axis.
# RESULTS[run][sensor][axis] = dict with keys: raw, clean, mask, z, bias

RESULTS = {}

print("=" * 70)
print(f"  Running spike removal (|z| > {SPIKE_Z_THRESHOLD})")
print("=" * 70)
print(f"\n  {'run':<6}{'sensor':<7}{'axis':<5}"
      f"{'spikes':>10}{'spike%':>10}{'bias':>14}{'sigma_after':>14}")
print("  " + "-" * 66)

for run in RUNS:
    RESULTS[run] = {}
    for sensor in SENSOR_FILES:
        if sensor not in DATA.get(run, {}):
            continue
        RESULTS[run][sensor] = {}
        for axis in AXES:
            v = DATA[run][sensor][axis].to_numpy()
            sigma_ref = SIGMA_REF[sensor][axis]
            v_clean, mask_spike, z, bias = clean_channel(v, sensor, axis, sigma_ref)
            RESULTS[run][sensor][axis] = {
                'raw':   v,
                'clean': v_clean,
                'mask':  mask_spike,
                'z':     z,
                'bias':  bias,
                'sigma_ref': sigma_ref,
            }
            n_spike = int(mask_spike.sum())
            pct = 100.0 * n_spike / len(v)
            # sigma of the cleaned signal (around its own median) — should be
            # close to sigma_ref for healthy runs, and much smaller than the
            # raw std for polluted runs.
            sigma_after = mad_sigma(v_clean - np.median(v_clean))
            print(f"  {run:<6}{sensor:<7}{axis:<5}"
                  f"{n_spike:>10,}{pct:>9.3f}%{bias:>14.4f}{sigma_after:>14.6f}")

In [ ]:
# Plot group A: overlay view with spike markers.
#
# One figure per run, arranged as a 3x3 grid (rows = sensor, cols = axis).
# Each subplot layers:
#   - grey thin line : raw signal
#   - red dots       : samples flagged as spikes (|z| > 5)
#   - blue line      : cleaned signal after linear interpolation
#
# This answers "did the detector catch the right samples?"

UNIT = {'gyro': 'rad/s', 'accel': 'm/s^2', 'mag': 'uT'}


def plot_overlay(run: str):
    fig, axes = plt.subplots(3, 3, figsize=(14, 8), sharex=True)
    fig.suptitle(f"Run {run} — raw (grey) + spikes (red) + clean (blue)",
                 y=0.995, fontsize=12)

    for i, sensor in enumerate(SENSOR_FILES):
        if sensor not in RESULTS.get(run, {}):
            for j in range(3):
                axes[i, j].text(0.5, 0.5, f"{sensor}: no data",
                                ha='center', va='center')
            continue
        t = DATA[run][sensor]['time_s'].to_numpy()
        for j, axis in enumerate(AXES):
            r = RESULTS[run][sensor][axis]
            ax = axes[i, j]
            ax.plot(t, r['raw'],   color='#888888', lw=0.5, label='raw')
            ax.plot(t, r['clean'], color='#1f77b4', lw=0.7, label='clean')
            if r['mask'].any():
                ax.plot(t[r['mask']], r['raw'][r['mask']],
                        'o', color='#E74C3C', ms=2.5, label='spike')
            n_sp = int(r['mask'].sum())
            pct  = 100.0 * n_sp / len(r['raw'])
            ax.set_title(f"{sensor}-{axis}   spikes={n_sp:,} ({pct:.3f}%)")
            ax.set_ylabel(UNIT[sensor])
            if i == 2:
                ax.set_xlabel('time (s)')
            if i == 0 and j == 2:
                ax.legend(loc='upper right', fontsize=8)

    plt.tight_layout()
    plt.show()


for run in RUNS:
    if run not in RESULTS or not RESULTS[run]:
        continue
    plot_overlay(run)

In [ ]:
# Plot group B: clean-only view (no spike markers).
#
# One figure per run, 3x3 grid (rows = sensor, cols = axis).
# Shows the cleaned signal alone so the residual noise shape is visible.
# Y-axis is auto-zoomed to [median - 4*sigma_ref, median + 4*sigma_ref]
# so that polluted runs don't get visually dominated by leftover outliers.


def plot_clean_only(run: str):
    fig, axes = plt.subplots(3, 3, figsize=(14, 8), sharex=True)
    fig.suptitle(f"Run {run} — cleaned signals", y=0.995, fontsize=12)

    for i, sensor in enumerate(SENSOR_FILES):
        if sensor not in RESULTS.get(run, {}):
            for j in range(3):
                axes[i, j].text(0.5, 0.5, f"{sensor}: no data",
                                ha='center', va='center')
            continue
        t = DATA[run][sensor]['time_s'].to_numpy()
        for j, axis in enumerate(AXES):
            r = RESULTS[run][sensor][axis]
            ax = axes[i, j]
            ax.plot(t, r['clean'], color=COLORS[axis], lw=0.6)

            sigma_ref = r['sigma_ref']
            med = float(np.median(r['clean']))
            ax.set_ylim(med - 4 * sigma_ref, med + 4 * sigma_ref)

            sigma_after = mad_sigma(r['clean'] - np.median(r['clean']))
            ax.set_title(
                f"{sensor}-{axis}   "
                f"sigma_after={sigma_after:.4g}  "
                f"sigma_ref={sigma_ref:.4g}"
            )
            ax.set_ylabel(UNIT[sensor])
            if i == 2:
                ax.set_xlabel('time (s)')

    plt.tight_layout()
    plt.show()


for run in RUNS:
    if run not in RESULTS or not RESULTS[run]:
        continue
    plot_clean_only(run)

In [ ]:
# Zoom-in on the most-polluted run to sanity-check the spike detector.
#
# We pick Run0011 and plot the first 2 seconds so individual spikes are
# resolvable. Same overlay as plot group A, just time-windowed.

ZOOM_RUN = '0011'
ZOOM_SECONDS = 2.0

if ZOOM_RUN in RESULTS and RESULTS[ZOOM_RUN]:
    fig, axes = plt.subplots(3, 3, figsize=(14, 8), sharex=True)
    fig.suptitle(
        f"Run {ZOOM_RUN} — first {ZOOM_SECONDS:g}s zoom-in",
        y=0.995, fontsize=12,
    )

    for i, sensor in enumerate(SENSOR_FILES):
        if sensor not in RESULTS[ZOOM_RUN]:
            for j in range(3):
                axes[i, j].text(0.5, 0.5, f"{sensor}: no data",
                                ha='center', va='center')
            continue
        t = DATA[ZOOM_RUN][sensor]['time_s'].to_numpy()
        sel = t <= ZOOM_SECONDS
        for j, axis in enumerate(AXES):
            r = RESULTS[ZOOM_RUN][sensor][axis]
            ax = axes[i, j]
            ax.plot(t[sel], r['raw'][sel],   color='#888888', lw=0.6, label='raw')
            ax.plot(t[sel], r['clean'][sel], color='#1f77b4', lw=0.9, label='clean')
            m = r['mask'] & sel
            if m.any():
                ax.plot(t[m], r['raw'][m], 'o', color='#E74C3C', ms=3.0,
                        label='spike')
            ax.set_title(f"{sensor}-{axis}")
            ax.set_ylabel(UNIT[sensor])
            if i == 2:
                ax.set_xlabel('time (s)')
            if i == 0 and j == 2:
                ax.legend(loc='upper right', fontsize=8)

    plt.tight_layout()
    plt.show()
else:
    print(f"Run {ZOOM_RUN} not loaded — skipping zoom-in.")